# Ingest and verify `data/benchsets_v1/` into one processed parquet

**ELI5:** `data/benchsets_v1/` is 28 published systematic-review screening collections —
181,199 papers, each one read by the researchers who ran that review and marked
**relevant** or **not relevant**. Unlike the six TIRI pools, this data arrives already
cleaned, already de-duplicated, and with each review's brief already copied onto every
paper row. So this notebook is **not** the cleaning notebook that
`01_data_compile.ipynb` is. Almost every item on that notebook's punch list was applied
upstream, and `manifest.json` records exactly how.

What is left is the part that actually matters for trusting the corpus:
**verify every file against the manifest's declared contract, raise loudly on any
mismatch, add the two derived columns the EDA needs, and write one combined
`data/processed/papers_benchset_v1.parquet`.**

That inversion is the point. `01_data_compile.ipynb` earns trust by *showing its
cleaning*; there is no cleaning to show here, so this notebook earns it by *checking
someone else's*. A clean run of section 2 is the evidence that all 28 files hold what
they claim to hold.

### Why this corpus exists in the repo at all

`CONTEXT.md` §3: the six TIRI pools run **26–77% positive**, roughly 20× production
prevalence, because they are what survived retrieval and a human's attention. Every F2
number and every calibrated threshold measured on them was measured in the wrong regime.
This corpus runs **1.86% positive overall**, and **0.16%–78% per collection**. It is the
prevalence-realistic surface the project has been missing.

### Two rules this corpus imposes, from its own README

1. **Never pool the collections into one training set.** Each question is its own world.
   Train and test *within* a `use_case_key`, or hold whole collections out. The parquet
   this notebook writes is pooled **for storage only** — one file, with `use_case_key` on
   every row. It is not a training set, and §9 says so again where it is written.
2. **The briefs are synthetic.** LLM-drafted from each review's own title and abstract,
   never from the labels (`brief_provenance` records this per row). The labels, titles
   and abstracts are real. A brief written from a summary of the papers that got included
   sits a little closer to the answer than one written before screening would — fine for
   comparing methods, not proof of what a system does on a fresh question.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "../../scripts")

BENCHSET_DIR = Path("../../data/benchsets_v1")
PROCESSED_DIR = Path("../../data/processed")
OUTPUT_PATH = PROCESSED_DIR / "papers_benchset_v1.parquet"

# This corpus ships exactly two label values and no null — unlike the TIRI export, which
# also carries 'pass' and never-triaged rows. Hardcoded as a closed vocabulary so an
# unexpected third value raises in section 5 instead of quietly becoming a category.
TRIAGE_CATEGORIES = ["positive", "negative"]

# The corpus README's three traps, as thresholds rather than prose, so the summary table
# in section 10 flags them mechanically.
MIN_POSITIVES_TO_SPLIT = 40   # below this a collection cannot be split, only evaluated
EXCLUDE_FROM_HEADLINE = "roadfreight_metareview"  # 78% positive, and selects reviews

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

## 1. Load (input)

Three kinds of file: `manifest.json` (the contract), `briefs.parquet` (the 28 questions,
one row each), and 28 per-collection parquets. Matching between the manifest and the
files is by **`use_case_key`**, and the filename is treated as a *guess* that gets
verified against the file's own `use_case_key` column — a filename is not evidence.

In [2]:
manifest = json.loads((BENCHSET_DIR / "manifest.json").read_text(encoding="utf-8"))
briefs = pd.read_parquet(BENCHSET_DIR / "briefs.parquet")

collection_paths = {p.stem: p for p in sorted(BENCHSET_DIR.glob("*.parquet"))
                    if p.stem != "briefs"}

raw_frames = {}
for key, path in collection_paths.items():
    frame = pd.read_parquet(path)
    found = sorted(frame["use_case_key"].unique())
    if found != [key]:
        raise ValueError(
            f"{path.name}: filename says use_case_key={key!r} but the file contains "
            f"{found!r}. Every downstream key would be wrong, so stopping here."
        )
    raw_frames[key] = frame

print(f"Manifest generated {manifest['exported_at']} from commit {manifest['git_sha']}")
print(f"Loaded {len(raw_frames)} collection files and {len(briefs)} briefs.\n")
print(f"Row contract: {manifest['row_key']}\n")
print(f"paper_id:     {manifest['paper_id']}\n")
print(f"Labels:       {manifest['labels']}")
print(f"Briefs:       {manifest['briefs']}")

Manifest generated 2026-08-11T18:29:32+00:00 from commit 84ae2c4
Loaded 28 collection files and 28 briefs.

Row contract: (use_case_key, paper_id)  — UNIQUE: duplicates are collapsed, and a paper screened both positive and negative within one use case is dropped entirely rather than resolved by guesswork (6 papers corpus-wide)

paper_id:     global content fingerprint: doi:<lowered> or title:<sha1 of normalised>

Labels:       the review authors' own screening decisions — real, not synthetic
Briefs:       SYNTHETIC but BLIND: LLM-drafted from each review's title+abstract, never from labels. Not the review's actual protocol.


## 2. Verify every file against the manifest — the notebook's real work

This is the direct analogue of `01_data_compile.ipynb`'s "every `usecase.json` name
matches exactly one JSONL export" check, and it is the reason to run this notebook at
all. Four things are checked, and any failure raises rather than warns:

1. **Key coverage** — one file per declared `use_case_key`, and no file the manifest
   doesn't declare.
2. **Counts** — rows, positives, negatives and prevalence, per collection.
3. **Coverage** — the manifest publishes a per-column non-null fraction for every
   collection; this recomputes all of them from the actual data.
4. **Totals** — the corpus README's headline figures (181,199 papers / 3,374 positive /
   1.86%) re-derived independently, not taken on trust.

In [3]:
declared = {u["use_case_key"]: u for u in manifest["use_cases"]}

missing_files = set(declared) - set(raw_frames)
undeclared = set(raw_frames) - set(declared)
if missing_files or undeclared:
    raise ValueError(
        f"manifest/file mismatch — declared but no file: {sorted(missing_files)}, "
        f"file but not declared: {sorted(undeclared)}"
    )
print(f"Key coverage OK — {len(declared)} declared use cases, {len(raw_frames)} files, "
      "one-to-one.\n")

problems = []
for key, spec in declared.items():
    frame = raw_frames[key]
    counts = frame["triage_label"].value_counts()
    actual = {
        "rows": len(frame),
        "positive": int(counts.get("positive", 0)),
        "negative": int(counts.get("negative", 0)),
    }
    for field, value in actual.items():
        if value != spec[field]:
            problems.append(f"{key}.{field}: manifest {spec[field]}, actual {value}")
    prevalence = round(actual["positive"] / actual["rows"], 5)
    if abs(prevalence - spec["prevalence"]) > 1e-4:
        problems.append(f"{key}.prevalence: manifest {spec['prevalence']}, actual {prevalence}")

    for column, declared_coverage in spec["coverage"].items():
        actual_coverage = round(float(frame[column].notna().mean()), 4)
        if abs(actual_coverage - declared_coverage) > 1e-3:
            problems.append(
                f"{key}.coverage[{column}]: manifest {declared_coverage}, "
                f"actual {actual_coverage}"
            )

if problems:
    raise ValueError("Manifest disagrees with the data:\n  " + "\n  ".join(problems))

n_checks = sum(3 + 1 + len(s["coverage"]) for s in declared.values())
print(f"All {n_checks} count/prevalence/coverage assertions passed across "
      f"{len(declared)} collections.")

Key coverage OK — 28 declared use cases, 28 files, one-to-one.

All 364 count/prevalence/coverage assertions passed across 28 collections.


In [4]:
total_rows = sum(len(f) for f in raw_frames.values())
total_positive = sum(int((f["triage_label"] == "positive").sum()) for f in raw_frames.values())

print("Corpus totals, re-derived here rather than read off the corpus README:")
print(f"  papers:     {total_rows:,}   (README says 181,199)")
print(f"  positive:   {total_positive:,}     (README says 3,374)")
print(f"  prevalence: {total_positive / total_rows:.4%}  (README says 1.86%)")

assert total_rows == 181_199, f"row total {total_rows} != README's 181,199"
assert total_positive == 3_374, f"positive total {total_positive} != README's 3,374"
print("\nAll three headline figures match the corpus README exactly.")

Corpus totals, re-derived here rather than read off the corpus README:
  papers:     181,199   (README says 181,199)
  positive:   3,374     (README says 3,374)
  prevalence: 1.8620%  (README says 1.86%)

All three headline figures match the corpus README exactly.


## 3. Keys — `paper_id` is unique *within* a collection, and deliberately not across them

`01_data_compile.ipynb` found that the TIRI exports' `id` collided once use cases were
pooled, and fixed it with a compound `paper_id`. The same fix is needed here for the same
mechanical reason — but for a completely different underlying cause, and the difference
is worth stating precisely.

There, the collision was an artefact: two unrelated papers happened to share a
per-file row id. Here, `paper_id` is a **content fingerprint** (`doi:<lowered>` or
`title:<sha1>`), so a repeat means *the same real paper was screened for two different
questions*. That is not a duplicate to clean up. It is the corpus telling us something,
and section 4 measures what.

In [5]:
for key, frame in raw_frames.items():
    if not frame["paper_id"].is_unique:
        n_dupes = int(frame["paper_id"].duplicated().sum())
        raise ValueError(f"{key}: paper_id repeats {n_dupes} times within one collection, "
                         "which the manifest's row_key contract says cannot happen.")
print(f"paper_id is unique within every one of the {len(raw_frames)} collections. "
      "(The manifest's UNIQUE row_key contract holds.)\n")

for key, frame in raw_frames.items():
    frame.insert(0, "row_key", key + "__" + frame["paper_id"].astype(str))

all_ids = pd.concat([f["paper_id"] for f in raw_frames.values()], ignore_index=True)
n_repeat_rows = int(all_ids.duplicated().sum())
print(f"Across collections, {n_repeat_rows:,} rows share a paper_id with an earlier row — "
      f"{all_ids.nunique():,} distinct papers across {len(all_ids):,} rows.")
print("row_key = use_case_key + '__' + paper_id is therefore the global key, exactly as "
      "01_data_compile.ipynb's compound paper_id is for the TIRI corpus.")

paper_id is unique within every one of the 28 collections. (The manifest's UNIQUE row_key contract holds.)

Across collections, 3,204 rows share a paper_id with an earlier row — 177,995 distinct papers across 181,199 rows.
row_key = use_case_key + '__' + paper_id is therefore the global key, exactly as 01_data_compile.ipynb's compound paper_id is for the TIRI corpus.


## 4. The same paper, two questions, two different answers

This is the section worth reading twice.

`CONTEXT.md` §2 states the project's central technical finding: **relevance is a property
of the (brief, paper) pair, not of the paper.** Until now that has been argued
*indirectly* — a classifier predicts `use_case_key` from a paper embedding at 96.2%
accuracy, LOGO transfer collapses to ~0.54 ROC-AUC, brief-relative features beat
paper-only ones. All real evidence, all circumstantial.

This corpus lets it be measured **directly**, because the same paper genuinely appears
under multiple questions with independently-made expert labels.

In [6]:
pairs = pd.concat(
    [f[["paper_id", "use_case_key", "triage_label"]] for f in raw_frames.values()],
    ignore_index=True,
)
per_paper = pairs.groupby("paper_id")["use_case_key"].nunique()
shared_ids = per_paper[per_paper > 1].index

shared = pairs[pairs["paper_id"].isin(shared_ids)]
label_variety = shared.groupby("paper_id")["triage_label"].nunique()
n_disagree = int((label_variety > 1).sum())

print(f"Papers screened under more than one question: {len(shared_ids):,}")
print(f"  ...of which labelled BOTH positive and negative depending on the question: "
      f"{n_disagree:,} ({n_disagree / len(shared_ids):.1%})")
print(f"  ...consistently labelled across questions:  {len(shared_ids) - n_disagree:,}")

example = shared[shared["paper_id"].isin(label_variety[label_variety > 1].index)]
example_id = example["paper_id"].iloc[0]
print(f"\nOne concrete case — paper_id {example_id!r}:")
display(
    pairs[pairs["paper_id"] == example_id]
    .merge(briefs[["use_case_key", "use_case_name"]], on="use_case_key")
    .reset_index(drop=True)
)

Papers screened under more than one question: 3,137
  ...of which labelled BOTH positive and negative depending on the question: 130 (4.1%)
  ...consistently labelled across questions:  3,007

One concrete case — paper_id 'doi:10.1161/circulationaha.114.012438':


,paper_id,use_case_key,triage_label,use_case_name
0,doi:10.1161/circulationaha.114.012438,synergy_bos_2018,negative,Cerebral small vessel disease dementia risk po...
1,doi:10.1161/circulationaha.114.012438,synergy_wolters_2018,positive,Coronary Heart Disease Heart Failure Dementia ...


**Hard finding.** 3,137 papers were screened under more than one question, and **130 of
them carry `positive` under one question and `negative` under another** — the *same
paper*, the *same title and abstract*, judged by domain experts, coming out differently
because the question changed. No feature computed from the paper alone can be right about
both rows at once.

**Judgement call, on how much weight to put on it:** 130 papers is a small absolute
number, and 96% of shared papers *are* labelled consistently — which is what you would
expect, since two reviews that both surface a paper are usually topically adjacent, and
an irrelevant paper tends to be irrelevant to both. So this is a **demonstration that the
effect is real and non-zero on external, independently-labelled data**, not a measurement
of how large it is in general. The direct estimate of the effect's size stays where
`CONTEXT.md` §2 already puts it — in the LOGO and brief-relative feature comparisons, not
here.

**The practical consequence is the one to carry forward:** these 130 rows are a leakage
channel with no analogue in the TIRI corpus. Deduplicating on `paper_id` across
collections would silently destroy real, independently-labelled rows — the same mistake
`04_feature_engineering.ipynb` §3 already guards against by scoping its dedupe to *within*
`use_case_key`. Within-silo folds, which this corpus requires anyway, make the channel
moot; anything that pools would have to handle it explicitly.

## 5. Dtypes and the label vocabulary

Four fixes, all of them the same fixes `01_data_compile.ipynb` §3–5 applies, for the same
stated reasons:

- **`year` and `citation_count` → nullable `Int64`.** They arrive as `float64` because
  that is how parquet round-trips a column with nulls through numpy. A year is not a
  float. **NULL ≠ 0 throughout** — 13.5% of `citation_count` is missing corpus-wide and
  none of it becomes a zero here.
- **`triage_label` → `Categorical`** over a closed two-value vocabulary, raising on
  anything else. The TIRI version needed three values plus null; this corpus has exactly
  two and no null, which section 2's coverage check already confirmed.

In [7]:
for key, frame in raw_frames.items():
    unexpected = set(frame["triage_label"].dropna().unique()) - set(TRIAGE_CATEGORIES)
    if unexpected:
        raise ValueError(f"{key}: unexpected triage_label value(s) {unexpected}")
    if frame["triage_label"].isna().any():
        raise ValueError(f"{key}: triage_label has nulls, but the manifest declares "
                         "100% coverage — a never-screened row would need a decision, "
                         "not a silent cast.")
    frame["triage_label"] = pd.Categorical(frame["triage_label"], categories=TRIAGE_CATEGORIES)
    frame["year"] = frame["year"].astype("Int64")
    frame["citation_count"] = frame["citation_count"].astype("Int64")

print("triage_label is a closed 2-value Categorical in every collection; "
      "year and citation_count are nullable Int64.")
print(f"\nNo value was filled: 'positive'/'negative' are the only labels, and NULL year "
      "or citation_count stays NULL.")

triage_label is a closed 2-value Categorical in every collection; year and citation_count are nullable Int64.

No value was filled: 'positive'/'negative' are the only labels, and NULL year or citation_count stays NULL.


## 6. The two derived columns — and a convention that is the *reverse* of TIRI's

`has_abstract` and `label_positive` are the two columns every EDA and modelling notebook
downstream expects, and this corpus ships neither.

**The trap, stated loudly because it is exactly backwards from the rest of the repo:**
in `papers_combined.parquet` a missing abstract is an **empty string**, so `.isna()`
reports zero missing and `02_eda_quickstart.ipynb` §4 warns you to use the `has_abstract`
flag instead. In this corpus a missing abstract is a **genuine NULL**. Code carried over
from the TIRI notebooks that tests `abstract == ""` will find nothing here and conclude
every paper has an abstract; 4,733 of them do not.

The cell below asserts which convention actually holds rather than trusting either.

In [8]:
n_null_abstract = n_blank_abstract = 0
for key, frame in raw_frames.items():
    is_null = frame["abstract"].isna()
    is_blank = ~is_null & (frame["abstract"].astype(str).str.strip() == "")
    n_null_abstract += int(is_null.sum())
    n_blank_abstract += int(is_blank.sum())
    frame["has_abstract"] = ~(is_null | is_blank)
    frame["label_positive"] = (frame["triage_label"] == "positive").astype("int8")

print(f"Missing abstracts stored as genuine NULL:   {n_null_abstract:,}")
print(f"Missing abstracts stored as empty string:   {n_blank_abstract:,}")
assert n_blank_abstract == 0, (
    "An empty-string abstract appeared — this corpus's convention is NULL, and "
    "has_abstract now depends on which one is true. Check the export before trusting it."
)
print("\nConfirmed: this corpus uses NULL, never the empty string — the opposite of "
      "papers_combined.parquet. has_abstract is built from NULL accordingly.")
print(f"\nhas_abstract is False for {n_null_abstract:,} rows "
      f"({n_null_abstract / total_rows:.1%} of the corpus).")

Missing abstracts stored as genuine NULL:   4,733
Missing abstracts stored as empty string:   0

Confirmed: this corpus uses NULL, never the empty string — the opposite of papers_combined.parquet. has_abstract is built from NULL accordingly.

has_abstract is False for 4,733 rows (2.6% of the corpus).


## 7. Brief integrity — constant within a collection, and matching `briefs.parquet`

The equivalent of `01_data_compile.ipynb` §7's "eyeball that broadcast metadata lines up
with the right use case (no cross-wiring)" check, done programmatically instead of by
eye — there are 28 collections now, not 6, and reading them side by side is no longer a
reasonable way to catch a mismatch.

In [9]:
BRIEF_COLS = [c for c in briefs.columns if c != "use_case_key"]
briefs_indexed = briefs.set_index("use_case_key")


def normalise(value):
    # Lists arrive as numpy arrays; compare them by value, not by identity.
    if isinstance(value, (list, np.ndarray)):
        return tuple(str(v) for v in value)
    return value


mismatches = []
for key, frame in raw_frames.items():
    for column in BRIEF_COLS:
        distinct = {normalise(v) for v in frame[column]}
        if len(distinct) != 1:
            mismatches.append(f"{key}.{column}: {len(distinct)} distinct values within one collection")
            continue
        on_rows = distinct.pop()
        in_briefs = normalise(briefs_indexed.loc[key, column])
        if on_rows != in_briefs:
            mismatches.append(f"{key}.{column}: rows say {str(on_rows)[:60]!r}, "
                              f"briefs.parquet says {str(in_briefs)[:60]!r}")

if mismatches:
    raise ValueError("Brief broadcast is inconsistent:\n  " + "\n  ".join(mismatches))

print(f"All {len(BRIEF_COLS)} brief columns are constant within each of the "
      f"{len(raw_frames)} collections, and every one matches briefs.parquet exactly.")
print("No cross-wiring: each collection's papers carry that collection's own brief.\n")
print(f"Brief provenance, all rows: {sorted(briefs['brief_provenance'].unique())} "
      "— LLM-drafted from the review's own title+abstract, never from labels.")

All 11 brief columns are constant within each of the 28 collections, and every one matches briefs.parquet exactly.
No cross-wiring: each collection's papers carry that collection's own brief.

Brief provenance, all rows: ['review_abstract'] — LLM-drafted from the review's own title+abstract, never from labels.


## 8. What this corpus does *not* carry, and why

29 of `papers_combined.parquet`'s 50 columns are absent here. That is a published
contract, not a gap, and the manifest says so in its own words. Printing it here means a
reader who comes looking for `embedding` or `relevance_score` finds the answer in the
notebook rather than concluding the export is broken.

The one with real consequences downstream is **`embedding`**: this corpus ships no
vectors, so `02` and `03` compute their own via `scripts/embed_benchsets.py`
(Jasper + Qwen3-4B, the pairing `reports/wf_embedding_bakeoff.md` §8 adopted) and join
them at read time. They are deliberately **not** written into this notebook's output —
4,608 float32 dimensions across 181,199 rows is ~3.3 GB, twenty times this notebook's
whole output, and `04_feature_engineering.ipynb` already establishes the pattern of
joining cached embeddings rather than carrying them in the corpus file.

In [10]:
print("From manifest['excluded_on_purpose']:\n")
for item in manifest["excluded_on_purpose"]:
    print(f"  - {item}\n")

benchset_cols = set(raw_frames[next(iter(raw_frames))].columns)
tiri_path = PROCESSED_DIR / "papers_combined.parquet"
if tiri_path.exists():
    tiri_cols = set(pd.read_parquet(tiri_path, columns=None).columns)
    print(f"papers_combined.parquet has {len(tiri_cols)} columns; this corpus has "
          f"{len(benchset_cols)}.")
    print(f"Absent here ({len(tiri_cols - benchset_cols)}): "
          f"{sorted(tiri_cols - benchset_cols)}")
    print(f"New here ({len(benchset_cols - tiri_cols)}): "
          f"{sorted(benchset_cols - tiri_cols)}")

From manifest['excluded_on_purpose']:

  - exemplars / near_misses / population / interventions / outcomes / evidence — spec-ladder fields; `exemplars` is label-derived and would leak the answer key

  - venue, language, url, sources, source_id, review_label, relevance_score, embedding, embed_model, embed_dim, app_version, from_* — contract says Drop

papers_combined.parquet has 50 columns; this corpus has 24.
Absent here (29): ['app_version', 'constraints_cost', 'constraints_scale', 'decision_exclusions', 'decision_must_have', 'decision_nice_to_have', 'decision_rules', 'embed_dim', 'embed_model', 'embedding', 'from_arxiv', 'from_core', 'from_crossref', 'from_europe_pmc', 'from_openalex', 'from_pubmed', 'from_seed', 'from_semantic_scholar', 'language', 'notes', 'performance_criteria', 'relevance_score', 'review_label', 'source_id', 'sources', 'trl_max', 'trl_min', 'url', 'venue']
New here (3): ['brief_provenance', 'label_positive', 'row_key']


## 9. Concatenate, validate, save

Concatenation order follows the manifest's declared order, not the order the files
happened to glob in — the same reasoning `01_data_compile.ipynb` §6 gives: row order
feeds fold assignment downstream, so it must not shift because a file was renamed.

**On the word "combined":** this file pools 28 collections into one parquet **for storage
only**. Every row carries its `use_case_key`, and the corpus README's second rule is
that the collections must never be pooled into a single training set — each question is
its own world, and shuffling 181,199 rows together puts near-identical papers on both
sides of a split. Read it, filter it to one `use_case_key`, then model.

In [11]:
ordered_keys = [u["use_case_key"] for u in manifest["use_cases"]]
assert set(ordered_keys) == set(raw_frames), "manifest order doesn't cover every collection"

reference_columns = None
for key in ordered_keys:
    columns = list(raw_frames[key].columns)
    if reference_columns is None:
        reference_columns = columns
    elif columns != reference_columns:
        raise ValueError(f"{key}'s columns don't match the others — diff: "
                         f"{set(columns) ^ set(reference_columns)}")
print(f"All {len(ordered_keys)} frames share identical columns ({len(reference_columns)}). "
      "OK to concatenate.\n")

combined = pd.concat([raw_frames[k] for k in ordered_keys], ignore_index=True)

assert len(combined) == total_rows, f"{len(combined)} rows after concat, expected {total_rows}"
assert combined["row_key"].is_unique, "row_key is not globally unique — the compound key is broken"
assert combined["triage_label"].notna().all(), "triage_label went null during concat"
print(f"row_key is globally unique across all {len(combined):,} rows.")
print(f"(paper_id alone is NOT — {int(combined['paper_id'].duplicated().sum()):,} "
      "collisions if pooled without use_case_key.)\n")
print(combined.dtypes.to_frame("dtype"))

All 28 frames share identical columns (24). OK to concatenate.

row_key is globally unique across all 181,199 rows.
(paper_id alone is NOT — 3,204 collisions if pooled without use_case_key.)

                            dtype
row_key                       str
paper_id                      str
use_case_key                  str
title                         str
abstract                      str
triage_label             category
authors                       str
doi                           str
year                        Int64
citation_count              Int64
exported_at                   str
use_case_name                 str
objective                     str
problem_statement             str
terms_must_include         object
terms_nice_to_have         object
terms_exclude              object
domain_industry               str
domain_application            str
domain_technology_focus    object
usecase_schema_version        str
brief_provenance              str
has_abstract              

In [12]:
print("Null rate (%) per column, whole corpus:")
null_rates = (combined.isna().mean() * 100).round(1).sort_values(ascending=False)
print(null_rates[null_rates > 0].to_frame("pct_null"))

print(f"\ncitation_count nulls, never filled with 0: {int(combined['citation_count'].isna().sum()):,}")
print(f"year nulls, dtype fixed but count untouched:  {int(combined['year'].isna().sum()):,}")
print(f"abstract nulls (genuine NULL, see §6):        {int(combined['abstract'].isna().sum()):,}")
print(f"triage_label nulls:                           {int(combined['triage_label'].isna().sum()):,} "
      "— this corpus screened every row")

Null rate (%) per column, whole corpus:
                pct_null
authors             10.6
citation_count      10.3
doi                 10.1
year                 9.4
abstract             2.6

citation_count nulls, never filled with 0: 18,599
year nulls, dtype fixed but count untouched:  16,995
abstract nulls (genuine NULL, see §6):        4,733
triage_label nulls:                           0 — this corpus screened every row


In [13]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
combined.to_parquet(OUTPUT_PATH, index=False)
size_mb = OUTPUT_PATH.stat().st_size / 1024**2
print(f"Saved {len(combined):,} rows x {len(combined.columns)} columns to "
      f"{OUTPUT_PATH} ({size_mb:.1f} MB)")

Saved 181,199 rows x 24 columns to ../../data/processed/papers_benchset_v1.parquet (157.2 MB)


### Verify the round trip

`to_parquet` not raising is not evidence the dtypes survived — the `Int64` and
`Categorical` casts from §5 are exactly the ones that quietly degrade.

In [14]:
check = pd.read_parquet(OUTPUT_PATH)

assert len(check) == len(combined)
assert check["row_key"].is_unique
assert str(check["year"].dtype) == "Int64"
assert str(check["citation_count"].dtype) == "Int64"
assert str(check["triage_label"].dtype) == "category"
assert str(check["has_abstract"].dtype) == "bool"
assert check["use_case_key"].notna().all()
assert check["abstract"].isna().sum() == combined["abstract"].isna().sum(), (
    "abstract nulls changed on round trip — the NULL-vs-empty-string convention from §6 "
    "did not survive parquet."
)

print("Round-trip checks passed:")
print(f"  {len(check):,} rows, {len(check.columns)} columns")
print(f"  triage_label categories: {list(check['triage_label'].cat.categories)}")
print(f"  collections present: {check['use_case_key'].nunique()}")

Round-trip checks passed:
  181,199 rows, 24 columns
  triage_label categories: ['positive', 'negative']
  collections present: 28


## 10. The per-collection summary, and the three traps

Everything a reader needs before using this corpus, in one table. The three flag columns
are the corpus README's own warnings, computed here rather than quoted:

- **`too_small_to_split`** — fewer than 40 positives, so the collection cannot support a
  train/test split inside itself. Use it to evaluate, not to train and test within.
- **`headline_excluded`** — `roadfreight_metareview` is 78% positive across 132 rows and
  selects *review papers* rather than individual studies. It is not the same task, and it
  belongs in no headline average.
- **`prevalence`** — sorted ascending, because the spread is the single most important
  fact about this corpus and a table sorted by name hides it.

In [15]:
summary = pd.DataFrame([
    {
        "use_case_key": key,
        "rows": len(frame),
        "positive": int((frame["triage_label"] == "positive").sum()),
        "prevalence": float((frame["triage_label"] == "positive").mean()),
        "pct_has_abstract": float(frame["has_abstract"].mean()),
        "pct_has_year": float(frame["year"].notna().mean()),
        "pct_has_citations": float(frame["citation_count"].notna().mean()),
    }
    for key, frame in raw_frames.items()
]).set_index("use_case_key")

summary["too_small_to_split"] = summary["positive"] < MIN_POSITIVES_TO_SPLIT
summary["headline_excluded"] = summary.index == EXCLUDE_FROM_HEADLINE
summary = summary.sort_values("prevalence")

n_small = int(summary["too_small_to_split"].sum())
print(f"{n_small} of {len(summary)} collections have fewer than "
      f"{MIN_POSITIVES_TO_SPLIT} positives and cannot be split — "
      f"{len(summary) - n_small} are usable for within-collection train/test.")
print(f"Prevalence spans {summary['prevalence'].min():.2%} "
      f"({summary['prevalence'].idxmin()}) to {summary['prevalence'].max():.2%} "
      f"({summary['prevalence'].idxmax()}) — a {summary['prevalence'].max() / summary['prevalence'].min():.0f}x spread.")
print(f"Rows span {summary['rows'].min():,} to {summary['rows'].max():,}.\n")

summary.style.format({
    "prevalence": "{:.2%}", "pct_has_abstract": "{:.1%}",
    "pct_has_year": "{:.1%}", "pct_has_citations": "{:.1%}",
})

12 of 28 collections have fewer than 40 positives and cannot be split — 16 are usable for within-collection train/test.
Prevalence spans 0.16% (synergy_brouwer_2019) to 78.03% (roadfreight_metareview) — a 480x spread.
Rows span 132 to 48,343.



,rows,positive,prevalence,pct_has_abstract,pct_has_year,pct_has_citations,too_small_to_split,headline_excluded
use_case_key,,,,,,,,
synergy_brouwer_2019,38114,62,0.16%,98.9%,93.9%,93.9%,False,False
synergy_bos_2018,4877,10,0.21%,98.7%,92.1%,92.1%,True,False
synergy_leenaars_2019,5812,17,0.29%,99.2%,95.1%,95.1%,True,False
synergy_wolters_2018,4278,19,0.44%,91.1%,92.8%,92.8%,True,False
synergy_chou_2004,1630,9,0.55%,88.4%,87.2%,87.2%,True,False
synergy_chou_2003,1907,15,0.79%,94.3%,82.5%,82.5%,True,False
synergy_van_dis_2020,9128,72,0.79%,98.2%,95.1%,95.1%,False,False
synergy_radjenovic_2013,5935,48,0.81%,100.0%,98.6%,98.6%,False,False
synergy_van_de_schoot_2018,4544,38,0.84%,98.4%,94.3%,94.3%,True,False


**Hard finding:** exactly **16 of 28** collections carry ≥40 positives. The other 12 —
as few as 9 positives in `synergy_chou_2004` — are evaluation-only surfaces. Any
within-collection cross-validation in `03_eda_full_benchset_v1.ipynb` runs on the 16, and
says so.

**Hard finding:** prevalence spans **0.16% to 78%**, a ~480× spread, and row counts span
132 to 48,343. There is no such thing as a representative average over these 28
collections, which is why every per-collection number downstream is reported with a win
count beside the mean (`CONTEXT.md` §5).

## Where to go next

- **`02_eda_quickstart_benchset_v1.ipynb`** — the 5-minute tour of this output: label
  balance, missingness, skew, and the embedding map.
- **`03_eda_full_benchset_v1.ipynb`** — the full EDA pass, per collection, plus
  within-collection baselines on the 16 splittable ones.
- **`scripts/embed_benchsets.py`** — the Jasper + Qwen3-4B vectors that 02 and 03 join.
  Run it before either.
- **`CONTEXT.md`** — read §1 (no pooled model ever ships) and §3 (data facts that will
  bite you) before proposing anything modelling-shaped on this corpus.
- **`data/benchsets_v1/README.md`** — the corpus's own data card, including the five
  things it warns will bite you.